# 03 — Plotly Visualization

**Topics covered:**
1. Line chart — time series
2. Bar chart — category comparison
3. Scatter plot — correlation
4. Histogram & box plot
5. Heatmap — correlation matrix
6. Pie / donut chart
7. Subplots — dashboard layout
8. Mini-exercises

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from ds_sandbox.dataset import load_iris, make_sales, make_timeseries

import plotly
print(f'Plotly version: {plotly.__version__}')

Plotly version: 5.9.0


## 1. Line Chart — Time Series

In [2]:
ts = make_timeseries()
ts['rolling_7'] = ts['value'].rolling(7).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts['date'], y=ts['value'],
                         mode='lines', name='Daily', opacity=0.4,
                         line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=ts['date'], y=ts['rolling_7'],
                         mode='lines', name='7-day MA',
                         line=dict(color='crimson', width=2)))

fig.update_layout(title='Time Series with 7-Day Moving Average',
                  xaxis_title='Date', yaxis_title='Value',
                  hovermode='x unified')
fig.show()

/opt/anaconda3/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



## 2. Bar Chart — Category Comparison

In [3]:
sales = make_sales()

region_rev = (sales.groupby('region')['revenue']
              .sum().reset_index()
              .sort_values('revenue', ascending=False))

fig = px.bar(region_rev, x='region', y='revenue',
             color='region', text_auto='.2s',
             title='Total Revenue by Region',
             color_discrete_sequence=px.colors.qualitative.Set2)
fig.update_traces(textposition='outside')
fig.show()

In [4]:
# Grouped bar — product x region
prod_region = (sales.groupby(['product', 'region'])['revenue']
               .sum().reset_index())

fig = px.bar(prod_region, x='region', y='revenue',
             color='product', barmode='group',
             title='Revenue by Region and Product')
fig.show()

## 3. Scatter Plot — Correlation

In [5]:
iris = load_iris()

fig = px.scatter(iris,
                 x='sepal length (cm)', y='sepal width (cm)',
                 color='species', size='petal length (cm)',
                 hover_data=['petal width (cm)'],
                 title='Iris: Sepal Length vs Width',
                 trendline='ols')
fig.show()

In [6]:
# Scatter matrix (pair plot)
fig = px.scatter_matrix(iris,
                        dimensions=['sepal length (cm)', 'sepal width (cm)',
                                    'petal length (cm)', 'petal width (cm)'],
                        color='species', title='Iris Scatter Matrix')
fig.update_traces(diagonal_visible=False)
fig.show()

AttributeError: 'DataFrame' object has no attribute 'iteritems'

## 4. Histogram & Box Plot

In [7]:
fig = px.histogram(iris, x='sepal length (cm)', color='species',
                   marginal='rug', nbins=25,
                   title='Distribution of Sepal Length')
fig.show()

In [8]:
fig = px.box(iris, x='species', y='petal length (cm)',
             color='species', points='all',
             title='Petal Length Distribution per Species')
fig.show()

## 5. Heatmap — Correlation Matrix

In [9]:
numeric_cols = ['sepal length (cm)', 'sepal width (cm)',
                'petal length (cm)', 'petal width (cm)']
corr = iris[numeric_cols].corr().round(2)

fig = px.imshow(corr,
                text_auto=True,
                color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1,
                title='Iris Feature Correlation Matrix')
fig.show()

## 6. Pie / Donut Chart

In [10]:
product_share = (sales.groupby('product')['revenue']
                 .sum().reset_index())

fig = px.pie(product_share, values='revenue', names='product',
             hole=0.4, title='Revenue Share by Product')
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

## 7. Subplots — Dashboard Layout

In [12]:
monthly_ts = (ts.set_index('date')['value']
              .resample('M').mean().reset_index())

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Monthly Avg Revenue', 'Region Revenue',
                    'Iris Sepal Scatter', 'Petal Length Distribution'],
    specs=[[{'type': 'scatter'}, {'type': 'bar'}],
           [{'type': 'scatter'}, {'type': 'box'}]]
)

# Top-left: monthly time series
fig.add_trace(go.Scatter(x=monthly_ts['date'], y=monthly_ts['value'],
                         mode='lines+markers', name='Monthly Avg'), row=1, col=1)

# Top-right: region bar
for r in region_rev.itertuples():
    fig.add_trace(go.Bar(x=[r.region], y=[r.revenue],
                         name=r.region, showlegend=False), row=1, col=2)

# Bottom-left: scatter
for sp in iris['species'].unique():
    sub = iris[iris['species'] == sp]
    fig.add_trace(go.Scatter(x=sub['sepal length (cm)'],
                             y=sub['sepal width (cm)'],
                             mode='markers', name=sp), row=2, col=1)

# Bottom-right: box
for sp in iris['species'].unique():
    sub = iris[iris['species'] == sp]
    fig.add_trace(go.Box(y=sub['petal length (cm)'],
                         name=sp, showlegend=False), row=2, col=2)

fig.update_layout(height=700, title_text='Data Scientist Sandbox — Dashboard')
fig.show()

/opt/anaconda3/lib/python3.11/site-packages/_plotly_utils/basevalidators.py:106: FutureWarning:

The behavior of DatetimeProperties.to_pydatetime is deprecated, in a future version this will return a Series containing python datetime objects instead of an ndarray. To retain the old behavior, call `np.array` on the result



## 8. Mini-Exercises

In [ ]:
# Exercise 1: Create a stacked bar chart of sales by month, coloured by product
# Your code here


In [ ]:
# Exercise 2: Create a violin plot comparing sepal widths across iris species
# Your code here


In [ ]:
# Exercise 3: Plot the daily time series with a shaded band ±1 std around the rolling mean
# Your code here
